##  1: Импорты, настройки, константы

In [1]:
# 1. Imports, settings, constants

import sys
import json
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# --- Make src.* importable from any working directory ---
_root = Path.cwd()
while not (_root / 'src').is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.paths import (
    get_data_path,
    get_base_path,
    get_project_root,
    get_genre_mapping_path,
)

# --- Timezone offset (0 = keep UTC) ---
UTC_OFFSET_HOURS = 0

# --- Constants ---
MIN_DURATION_MS = 10_000
START_YEAR, END_YEAR = 2020, 2026
SIGNIFICANCE_COUNT = 3
SIGNIFICANCE_COUNT_LOW = 2
SIGNIFICANCE_FRAC = 0.7

# --- Paths ---
PROJECT_ROOT = get_project_root()
DATA_DIR = get_data_path()
BASE_PATH = get_base_path()
GENRE_MAPPING_PATH = get_genre_mapping_path()

# --- Source files: history ---
HISTORY_GLOB = str(
    DATA_DIR / 'Spotify' / 'Spotify_ Extended_Streaming_History'
    / '*' / 'Streaming_History_Audio_*.json'
)
history_files = glob.glob(HISTORY_GLOB)

# --- Source files: Exportify playlists ---
EXPORTIFY_GLOB = str(
    DATA_DIR / 'Exportify' / 'spotify_playlists (*)' / '*.csv'
)
exportify_files = glob.glob(EXPORTIFY_GLOB)

# --- Source files: orphans ---
ORPHANS_DIR = DATA_DIR / 'Exportify' / 'exportify_orphan_export'
ORPHANS_PARTS = sorted(ORPHANS_DIR.glob('*.csv'))


def has_cyrillic(text) -> bool:
    """Check whether a string contains Cyrillic characters."""
    return bool(re.search(r'[а-яА-ЯёЁ]', str(text)))


def normalize_genres(genres_str, mapping):
    """Normalize a genre string via genre_mapping.json."""
    if pd.isna(genres_str) or genres_str == '':
        return 'unknown'
    parts = [g.strip() for g in str(genres_str).replace(';', ',').split(',') if g.strip()]
    if not parts:
        return 'unknown'
    normalized = []
    for g in parts:
        g_lower = g.lower()
        translated = mapping.get(g_lower)
        if translated:
            normalized.append(translated)
        elif has_cyrillic(g_lower):
            normalized.append('unknown')
        else:
            normalized.append(g)
    return ', '.join(normalized) if normalized else 'unknown'


# --- Load genre mapping once for the whole notebook ---
with open(GENRE_MAPPING_PATH, 'r', encoding='utf-8') as f:
    genre_mapping = json.load(f)

# --- Summary ---
print(f'History files:   {len(history_files)}')
print(f'Accounts:        {len({Path(f).parent.name for f in history_files})}')
print(f'Exportify CSVs:  {len(exportify_files)}')
print(f'Orphan parts:    {len(ORPHANS_PARTS)}')
print(f'Genre mappings:  {len(genre_mapping)}')
print(f'UTC offset:      {UTC_OFFSET_HOURS}')

History files:   15
Accounts:        3
Exportify CSVs:  364
Orphan parts:    8
Genre mappings:  568
UTC offset:      0


In [2]:
# 2. Load and clean history

import pandas as pd
from pathlib import Path

# --- Read all history JSON files ---
dfs = []
file_counts = []
for f in history_files:
    account = Path(f).parent.name
    try:
        df = pd.read_json(f)
        df['account'] = account
        dfs.append(df)
        file_counts.append(len(df))
    except Exception:
        file_counts.append(None)
        continue

if not dfs:
    raise ValueError('No history files loaded.')

df_hist = pd.concat(dfs, ignore_index=True)
raw_count = len(df_hist)

# --- Rename fields to unified schema ---
rename_map = {
    'endTime': 'ts',
    'master_metadata_track_name': 'track_name',
    'master_metadata_album_artist_name': 'artist_name',
    'master_metadata_album_album_name': 'album_name',
}
for old, new in rename_map.items():
    if old in df_hist.columns:
        df_hist.rename(columns={old: new}, inplace=True)

if 'ts' not in df_hist.columns:
    raise KeyError('Column ts not found after rename')

# --- Time features ---
df_hist['ts_utc'] = pd.to_datetime(df_hist['ts'], utc=True).dt.tz_localize(None)
df_hist['ts'] = df_hist['ts_utc'] + pd.Timedelta(hours=UTC_OFFSET_HOURS)

df_hist['year']    = df_hist['ts'].dt.year
df_hist['month']   = df_hist['ts'].dt.month
df_hist['day']     = df_hist['ts'].dt.day
df_hist['hour']    = df_hist['ts'].dt.hour
df_hist['weekday'] = df_hist['ts'].dt.weekday

# --- Skip flag ---
df_hist['is_skip'] = (df_hist['ms_played'] < MIN_DURATION_MS) | (df_hist.get('skipped', False))

# --- Cleaning ---
before = len(df_hist)
df_hist = df_hist.dropna(subset=['track_name', 'artist_name'])
dropped_na = before - len(df_hist)

before = len(df_hist)
df_hist = df_hist.drop_duplicates(subset=['spotify_track_uri', 'ts', 'account'])
dropped_dup = before - len(df_hist)

df_hist = df_hist[(df_hist['year'] >= START_YEAR) & (df_hist['year'] <= END_YEAR)]

# --- Drop podcast / audiobook / private columns ---
cols_to_drop = [
    # podcasts / audiobooks
    'episode_name', 'episode_show_name', 'spotify_episode_uri',
    'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title',
    # privacy
    'ip_addr', 'offline_timestamp', 'incognito_mode', 'conn_country',
]
df_hist = df_hist.drop(columns=[c for c in cols_to_drop if c in df_hist.columns])

# --- Save ---
df_hist.to_parquet(BASE_PATH / 'cleaned_history.parquet', index=False)

# --- Summary ---
print(f'History files: {len(history_files)}')
for idx, cnt in enumerate(file_counts, 1):
    mark = '✓' if cnt is not None else '✗'
    print(f'   [{idx}/{len(history_files)}] {mark} {cnt if cnt is not None else "—"}')

print(f'\nRaw events:        {raw_count}')
print(f'Dropped (no name): {dropped_na}')
print(f'Dropped (dupes):   {dropped_dup}')
print(f'Final events:      {len(df_hist)}')
print(f'Accounts:          {df_hist["account"].nunique()}')
print(f'Years:             {df_hist["year"].min()}–{df_hist["year"].max()}')
print(f'Skipped events:    {int(df_hist["is_skip"].sum())}')
print(f'Saved:             cleaned_history.parquet')

print(df_hist.dtypes)

History files: 15
   [1/15] ✓ 8306
   [2/15] ✓ 11556
   [3/15] ✓ 10121
   [4/15] ✓ 16183
   [5/15] ✓ 1582
   [6/15] ✓ 7899
   [7/15] ✓ 1829
   [8/15] ✓ 14111
   [9/15] ✓ 2187
   [10/15] ✓ 1196
   [11/15] ✓ 459
   [12/15] ✓ 116
   [13/15] ✓ 5
   [14/15] ✓ 4088
   [15/15] ✓ 417

Raw events:        80055
Dropped (no name): 25
Dropped (dupes):   261
Final events:      79769
Accounts:          3
Years:             2020–2026
Skipped events:    28011
Saved:             cleaned_history.parquet
ts                   datetime64[ns]
platform                     object
ms_played                     int64
track_name                   object
artist_name                  object
album_name                   object
spotify_track_uri            object
reason_start                 object
reason_end                   object
shuffle                        bool
skipped                        bool
offline                        bool
account                      object
ts_utc               datetime64[ns]
year 

In [3]:
# 3. Load and unify Exportify playlists

import pandas as pd
from pathlib import Path

# --- Read all Exportify CSV files ---
exportify_dfs = []
skipped_empty = 0
skipped_technical = 0

for f in exportify_files:
    fname = Path(f).name
    account = Path(f).parent.name

    # Skip the technical file with empty stem
    if Path(f).stem == '':
        skipped_technical += 1
        continue

    # Skip empty files
    if Path(f).stat().st_size == 0:
        skipped_empty += 1
        continue

    try:
        df = pd.read_csv(f)
        df['playlist_name'] = fname.replace('.csv', '', 1)
        df['account'] = account
        exportify_dfs.append(df)
    except Exception:
        continue

if not exportify_dfs:
    raise ValueError('No Exportify CSV files loaded.')

df_exportify_raw = pd.concat(exportify_dfs, ignore_index=True)
raw_rows = len(df_exportify_raw)

# --- Deduplicate tracks by URI ---
before = len(df_exportify_raw)
df_tracks = df_exportify_raw.drop_duplicates(subset=['Track URI'], keep='first').copy()
dropped_dup_tracks = before - len(df_tracks)

# --- Rename to unified schema ---
rename_exportify = {
    'Track URI': 'spotify_track_uri',
    'Track Name': 'track_name',
    'Artist Name(s)': 'artist_name',
    'Album Name': 'album_name',
    'Valence': 'valence',
    'Energy': 'energy',
    'Danceability': 'danceability',
    'Tempo': 'tempo',
    'Acousticness': 'acousticness',
    'Instrumentalness': 'instrumentalness',
    'Liveness': 'liveness',
    'Speechiness': 'speechiness',
    'Loudness': 'loudness',
    'Key': 'key',
    'Mode': 'mode',
    'Explicit': 'explicit',
    'Genres': 'genres',
    'Popularity': 'popularity',
    'Duration (ms)': 'duration_ms',
    'Release Date': 'release_date',
}
df_tracks.rename(columns=rename_exportify, inplace=True)

# --- Drop rows with empty spotify_track_uri ---
before = len(df_tracks)
df_tracks = df_tracks[df_tracks['spotify_track_uri'].notna()].copy()
dropped_no_uri = before - len(df_tracks)

# --- Fill missing genres and normalize ---
df_tracks['genres'] = df_tracks['genres'].fillna('unknown')
df_tracks['genres_normalized'] = df_tracks['genres'].apply(
    lambda x: normalize_genres(x, genre_mapping)
)
df_tracks['genres_normalized'] = df_tracks['genres_normalized'].fillna('unknown')

# --- Service fields ---
df_tracks['has_audio_features'] = True
df_tracks['source'] = 'exportify'
df_tracks['Added By'] = df_tracks['Added By'].fillna('unknown').astype(str)

# --- Save ---
df_tracks.to_parquet(BASE_PATH / 'tracks_exportify.parquet', index=False)

# --- Summary ---
unknown_count = (df_tracks['genres_normalized'] == 'unknown').sum()

print(f'Exportify CSVs:       {len(exportify_files)}')
print(f'Skipped (empty):      {skipped_empty}')
print(f'Skipped (technical):  {skipped_technical}')
print(f'Raw rows:             {raw_rows}')
print(f'Dropped (dupes):      {dropped_dup_tracks}')
print(f'Dropped (no URI):     {dropped_no_uri}')
print(f'Unique tracks:        {len(df_tracks)}')
print(f'Unknown genres:       {unknown_count} ({unknown_count / len(df_tracks) * 100:.1f}%)')
print(f'Saved:                tracks_exportify.parquet')

print(df_tracks.dtypes)

Exportify CSVs:       364
Skipped (empty):      0
Skipped (technical):  0
Raw rows:             40657
Dropped (dupes):      15703
Dropped (no URI):     1
Unique tracks:        24953
Unknown genres:       8624 (34.6%)
Saved:                tracks_exportify.parquet
spotify_track_uri      object
track_name             object
album_name             object
artist_name            object
release_date           object
duration_ms           float64
popularity            float64
explicit               object
Added By               object
Added At               object
genres                 object
Record Label           object
danceability          float64
energy                float64
key                   float64
loudness              float64
mode                  float64
speechiness           float64
acousticness          float64
instrumentalness      float64
liveness              float64
valence               float64
tempo                 float64
Time Signature        float64
playlist_name   

In [4]:
# 3.1. Build playlist_links.parquet

# --- df_exportify_raw must come from cell 3 ---
if 'df_exportify_raw' not in locals():
    raise ValueError('df_exportify_raw not found. Run cell 3 first.')

# --- Select link columns ---
playlist_links_cols = ['Track URI', 'Added By', 'Added At', 'playlist_name', 'account']
available_cols = [c for c in playlist_links_cols if c in df_exportify_raw.columns]
missing_cols = [c for c in playlist_links_cols if c not in df_exportify_raw.columns]

playlist_links = df_exportify_raw[available_cols].copy()

# --- Drop rows with empty Track URI ---
before = len(playlist_links)
playlist_links = playlist_links[playlist_links['Track URI'].notna()].copy()
dropped_no_uri = before - len(playlist_links)

# --- Cast Added By to string ---
if 'Added By' in playlist_links.columns:
    playlist_links['Added By'] = playlist_links['Added By'].fillna('unknown').astype(str)

# --- Save ---
playlist_links.to_parquet(BASE_PATH / 'playlist_links.parquet', index=False)

# --- Summary ---
print(f'Rows:              {len(playlist_links)}')
print(f'Dropped (no URI):  {dropped_no_uri}')
if missing_cols:
    print(f'Missing columns:   {missing_cols}')
print(f'Saved:             playlist_links.parquet')

print(playlist_links.dtypes)

Rows:              40647
Dropped (no URI):  10
Saved:             playlist_links.parquet
Track URI        object
Added By         object
Added At         object
playlist_name    object
account          object
dtype: object


In [5]:
# 4. First merge: history + Exportify tracks

# --- Load saved parquets ---
df_hist = pd.read_parquet(BASE_PATH / 'cleaned_history.parquet')
df_tracks = pd.read_parquet(BASE_PATH / 'tracks_exportify.parquet')

# --- LEFT JOIN history with tracks by URI ---
df_merged = df_hist.merge(
    df_tracks, on='spotify_track_uri', how='left', suffixes=('', '_track')
)

has_features = df_merged['has_audio_features'] == True

# --- Fill missing track fields from track data ---
for col in ['track_name', 'artist_name', 'album_name']:
    if col + '_track' in df_merged.columns:
        df_merged[col] = df_merged[col].fillna(df_merged[col + '_track'])
        df_merged.drop(columns=[col + '_track'], inplace=True)

# --- Drop remaining _track columns ---
cols_to_drop = [c for c in df_merged.columns if c.endswith('_track')]
if cols_to_drop:
    df_merged = df_merged.drop(columns=cols_to_drop)

# --- Save ---
df_merged.to_parquet(BASE_PATH / 'history_with_exportify.parquet', index=False)

# --- Summary ---
print(f'History events:    {len(df_hist)}')
print(f'Tracks:            {len(df_tracks)}')
print(f'Merged events:     {len(df_merged)}')
print(f'With features:     {int(has_features.sum())}')
print(f'Orphans (no feat): {int((~has_features).sum())}')
print(f'Saved:             history_with_exportify.parquet')

print(df_merged.dtypes)

History events:    79769
Tracks:            24953
Merged events:     79769
With features:     47975
Orphans (no feat): 31794
Saved:             history_with_exportify.parquet
ts                    datetime64[ns]
platform                      object
ms_played                      int64
track_name                    object
artist_name                   object
album_name                    object
spotify_track_uri             object
reason_start                  object
reason_end                    object
shuffle                         bool
skipped                         bool
offline                         bool
account                       object
ts_utc                datetime64[ns]
year                           int32
month                          int32
day                            int32
hour                           int32
weekday                        int32
is_skip                         bool
release_date                  object
duration_ms                  float64
popularity 

In [6]:
# 5. Load reference orphans

# --- Read orphan CSV parts ---
orphan_parts = []
for part_path in ORPHANS_PARTS:
    if not part_path.exists():
        continue
    df_part = pd.read_csv(part_path)
    orphan_parts.append(df_part)

if not orphan_parts:
    raise FileNotFoundError('No orphan CSV files loaded.')

df_orphans_raw = pd.concat(orphan_parts, ignore_index=True)

# --- Deduplicate by Track URI ---
before = len(df_orphans_raw)
df_orphans_raw = df_orphans_raw.drop_duplicates(subset=['Track URI'], keep='first')
dropped_dup = before - len(df_orphans_raw)

# --- Rename to unified schema ---
rename_exportify = {
    'Track URI': 'spotify_track_uri',
    'Track Name': 'track_name',
    'Artist Name(s)': 'artist_name',
    'Album Name': 'album_name',
    'Valence': 'valence',
    'Energy': 'energy',
    'Danceability': 'danceability',
    'Tempo': 'tempo',
    'Acousticness': 'acousticness',
    'Instrumentalness': 'instrumentalness',
    'Liveness': 'liveness',
    'Speechiness': 'speechiness',
    'Loudness': 'loudness',
    'Key': 'key',
    'Mode': 'mode',
    'Explicit': 'explicit',
    'Genres': 'genres',
    'Popularity': 'popularity',
    'Duration (ms)': 'duration_ms',
    'Release Date': 'release_date',
}
df_orphans = df_orphans_raw.rename(columns=rename_exportify)

# --- Keep only unified columns ---
needed_cols = list(rename_exportify.values())
df_orphans = df_orphans[[c for c in needed_cols if c in df_orphans.columns]].copy()

# --- Fill missing genres and normalize ---
df_orphans['genres'] = df_orphans['genres'].fillna('unknown')
df_orphans['genres_normalized'] = df_orphans['genres'].apply(
    lambda x: normalize_genres(x, genre_mapping)
)
df_orphans['genres_normalized'] = df_orphans['genres_normalized'].fillna('unknown')

# --- Service fields ---
df_orphans['has_audio_features'] = True
df_orphans['source'] = 'exportify_orphans'

# --- Save ---
df_orphans.to_parquet(BASE_PATH / 'orphans_unified.parquet', index=False)

# --- Summary ---
unknown_count = (df_orphans['genres_normalized'] == 'unknown').sum()

print(f'Parts loaded:      {len(orphan_parts)}')
print(f'Raw orphans:       {len(df_orphans_raw) + dropped_dup}')
print(f'Dropped (dupes):   {dropped_dup}')
print(f'Unique orphans:    {len(df_orphans)}')
print(f'Unknown genres:    {unknown_count} ({unknown_count / len(df_orphans) * 100:.1f}%)')
print(f'Saved:             orphans_unified.parquet')

print(df_orphans.dtypes)

Parts loaded:      8
Raw orphans:       18743
Dropped (dupes):   1
Unique orphans:    18742
Unknown genres:    8595 (45.9%)
Saved:             orphans_unified.parquet
spotify_track_uri      object
track_name             object
artist_name            object
album_name             object
valence               float64
energy                float64
danceability          float64
tempo                 float64
acousticness          float64
instrumentalness      float64
liveness              float64
speechiness           float64
loudness              float64
key                   float64
mode                  float64
explicit                 bool
genres                 object
popularity              int64
duration_ms             int64
release_date           object
genres_normalized      object
has_audio_features       bool
source                 object
dtype: object


In [7]:
# 5.1. Verify orphan genres

orphans = pd.read_parquet(BASE_PATH / 'orphans_unified.parquet')

total = len(orphans)
unknown = (orphans['genres_normalized'] == 'unknown').sum()
has_genres = orphans['genres_normalized'].notna() & (orphans['genres_normalized'].str.strip() != '')
cyrillic_count = orphans['genres_normalized'].apply(has_cyrillic).sum()

print(f'Orphans:              {total}')
print(f'Unknown genres:       {unknown} ({unknown / total * 100:.1f}%)')
print(f'Non-empty normalized: {has_genres.sum()} ({has_genres.mean() * 100:.1f}%)')
print(f'With Cyrillic:        {cyrillic_count}')

if has_genres.any():
    genre_counts = (
        orphans[orphans['genres_normalized'] != 'unknown']['genres_normalized']
        .str.split(',').explode().str.strip().value_counts()
    )
    print('\nTop-10 genres:')
    print(genre_counts.head(10).to_string())

Orphans:              18742
Unknown genres:       8595 (45.9%)
Non-empty normalized: 18742 (100.0%)
With Cyrillic:        0

Top-10 genres:
genres_normalized
downtempo         1555
trip hop          1419
organic house      779
coldwave           717
darkwave           583
melodic techno     559
post-punk          547
melodic house      540
ambient            485
idm                464


In [8]:
# 6. Second merge: enrich orphans

# --- Load intermediates ---
df_merged = pd.read_parquet(BASE_PATH / 'history_with_exportify.parquet')
df_orphans = pd.read_parquet(BASE_PATH / 'orphans_unified.parquet')

# --- Audio columns to fill for orphans ---
audio_cols = [
    'valence', 'energy', 'danceability', 'tempo',
    'acousticness', 'instrumentalness', 'liveness', 'speechiness',
    'loudness', 'key', 'mode', 'explicit',
    'genres', 'genres_normalized', 'popularity', 'duration_ms', 'release_date'
]

missing = [c for c in audio_cols if c not in df_orphans.columns]
if missing:
    raise ValueError(f'Orphans missing columns: {missing}')

# --- URI -> audio values lookup ---
orphan_lookup = df_orphans.set_index('spotify_track_uri')

# --- Identify orphans ---
is_orphan = df_merged['has_audio_features'] != True

# --- Fill audio columns for orphans ---
for col in audio_cols:
    df_merged.loc[is_orphan, col] = (
        df_merged.loc[is_orphan, 'spotify_track_uri'].map(orphan_lookup[col])
    )

# --- Update service fields ---
df_merged.loc[is_orphan, 'has_audio_features'] = True
df_merged.loc[is_orphan, 'source'] = 'exportify_orphans'

# --- Check remaining missing ---
still_missing = df_merged['has_audio_features'] != True

# --- Save ---
df_merged.to_parquet(BASE_PATH / 'history_fully_enriched.parquet', index=False)

# --- Summary ---
print(f'History events:     {len(df_merged)}')
print(f'Was orphans:        {int(is_orphan.sum())}')
print(f'Still missing feat: {int(still_missing.sum())}')
print(f'Saved:              history_fully_enriched.parquet')
print(f'Orphans reference:  {len(df_orphans)}')
print(df_merged.dtypes)

History events:     79769
Was orphans:        31794
Still missing feat: 0
Saved:              history_fully_enriched.parquet
Orphans reference:  18742
ts                    datetime64[ns]
platform                      object
ms_played                      int64
track_name                    object
artist_name                   object
album_name                    object
spotify_track_uri             object
reason_start                  object
reason_end                    object
shuffle                         bool
skipped                         bool
offline                         bool
account                       object
ts_utc                datetime64[ns]
year                           int32
month                          int32
day                            int32
hour                           int32
weekday                        int32
is_skip                         bool
release_date                  object
duration_ms                  float64
popularity                   float6

In [9]:
# 7. Categorize orphans and build full_history.parquet

# --- Load enriched history ---
df = pd.read_parquet(BASE_PATH / 'history_fully_enriched.parquet')

# --- 7.0 Clean technical noise ---
bad_mask = df['duration_ms'] == 0
if bad_mask.any():
    df = df[~bad_mask].copy()

tempo_zero = (df['tempo'] == 0).sum()
if tempo_zero > 0:
    df.loc[df['tempo'] == 0, 'tempo'] = np.nan

popularity_zero = (df['popularity'] == 0).sum()
if popularity_zero > 0:
    df.loc[df['popularity'] == 0, 'popularity'] = np.nan

# --- 7.1 Aggregate orphan metrics ---
is_orphan = df['source'] == 'exportify_orphans'

orphan_metrics = df[is_orphan].groupby('spotify_track_uri').agg(
    listen_count=('ts', 'count'),
    max_played_ms=('ms_played', 'max'),
    total_played_ms=('ms_played', 'sum'),
    first_listen=('ts', 'min'),
    last_listen=('ts', 'max')
).reset_index()

df_orphan_part = df[is_orphan].merge(orphan_metrics, on='spotify_track_uri', how='left')
df_not_orphan = df[~is_orphan].copy()

for col in orphan_metrics.columns:
    if col != 'spotify_track_uri':
        df_not_orphan[col] = np.nan

df = pd.concat([df_not_orphan, df_orphan_part], ignore_index=True)
is_orphan = df['source'] == 'exportify_orphans'

# --- 7.2 Events without audio features ---
no_valence_mask = df['valence'].isna()
df['orphan_category'] = 'not_orphan'

if no_valence_mask.any():
    df.loc[no_valence_mask, 'has_audio_features'] = False
    df.loc[no_valence_mask, 'orphan_category'] = 'unknown'

# --- 7.3 Categorize orphans ---
mask_orphan_with_features = is_orphan & ~no_valence_mask
mask_skipped = mask_orphan_with_features & df['is_skip']
df.loc[mask_skipped, 'orphan_category'] = 'skipped'

mask_rest = mask_orphan_with_features & (df['orphan_category'] == 'not_orphan')
played_frac = df['max_played_ms'] / df['duration_ms']

significant_condition = (
    (df['listen_count'] >= SIGNIFICANCE_COUNT) |
    ((df['listen_count'] >= SIGNIFICANCE_COUNT_LOW) & (played_frac >= SIGNIFICANCE_FRAC))
)
mask_significant = mask_rest & significant_condition
df.loc[mask_significant, 'orphan_category'] = 'significant'

mask_insignificant = mask_rest & ~significant_condition
df.loc[mask_insignificant, 'orphan_category'] = 'insignificant'

# --- 7.4 Fill missing genres ---
if df['genres'].isna().any():
    df['genres'] = df['genres'].fillna('unknown')

# --- 7.5 Drop leftovers ---
cols_to_drop = [c for c in df.columns if c in ['playlist_name', 'Time Signature', 'account_track']]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)

if 'genres_normalized' in df.columns and df['genres_normalized'].isna().any():
    df['genres_normalized'] = df['genres_normalized'].fillna('unknown')

# --- 7.6 Save ---
df.to_parquet(BASE_PATH / 'full_history.parquet', index=False)

# --- Summary ---
print(f'Events total:        {len(df)}')
print(f'Duration==0 dropped: {int(bad_mask.sum())}')
print(f'Tempo==0 -> NaN:     {int(tempo_zero)}')
print(f'Popularity==0 -> NaN:{int(popularity_zero)}')
print(f'\norphan_category:')
print(df['orphan_category'].value_counts().to_string())

print(f'\nNaN checks:')
print(f'   valence NaN:         {int(df["valence"].isna().sum())}')
print(f'   genres NaN:          {int(df["genres"].isna().sum())}')
print(f'   orphan_category NaN: {int(df["orphan_category"].isna().sum())}')
print(f'   duration_ms == 0:    {int((df["duration_ms"] == 0).sum())}')

print(f'\nSaved: full_history.parquet')

print(df.dtypes)
# --- Consistency checks ---
not_orphan_with_first = df[(df['orphan_category'] == 'not_orphan') & (df['first_listen'].notna())]
orphan_without_first = df[(df['source'] == 'exportify_orphans') & (df['first_listen'].isna())]

print(f'\nfirst_listen set on not_orphan:     {len(not_orphan_with_first)} (expected 0)')
print(f'first_listen missing on orphans:    {len(orphan_without_first)} (expected 0)')


Events total:        79752
Duration==0 dropped: 17
Tempo==0 -> NaN:     51
Popularity==0 -> NaN:17206

orphan_category:
orphan_category
not_orphan       47962
skipped          13033
significant      11559
insignificant     7191
unknown              7

NaN checks:
   valence NaN:         7
   genres NaN:          0
   orphan_category NaN: 0
   duration_ms == 0:    0

Saved: full_history.parquet
ts                    datetime64[ns]
platform                      object
ms_played                      int64
track_name                    object
artist_name                   object
album_name                    object
spotify_track_uri             object
reason_start                  object
reason_end                    object
shuffle                         bool
skipped                         bool
offline                         bool
account                       object
ts_utc                datetime64[ns]
year                           int32
month                          int32
day        

In [10]:
# 8. Build eda_ready.parquet

# --- Load full history ---
df_full = pd.read_parquet(BASE_PATH / 'full_history.parquet')

# --- Add played_frac ---
df_full['played_frac'] = df_full['ms_played'] / df_full['duration_ms']

# --- Drop podcast / audiobook columns (safety net) ---
cols_to_drop = [
    'episode_name', 'episode_show_name', 'spotify_episode_uri',
    'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title'
]
existing_to_drop = [c for c in cols_to_drop if c in df_full.columns]
if existing_to_drop:
    df_full = df_full.drop(columns=existing_to_drop)

# --- Save ---
df_full.to_parquet(BASE_PATH / 'eda_ready.parquet', index=False)
# --- Финальная проверка: не осталось ли колонок-артефактов после merge ---
# Суффиксы _x / _y появляются, когда pandas объединяет таблицы
# с одинаковыми именами колонок. Если такие есть — значит,
# какой-то merge не был обработан.
suspicious_cols = [c for c in df_full.columns if c.endswith('_x') or c.endswith('_y')]
if suspicious_cols:
    print(f'⚠️ Conflicting columns found: {suspicious_cols}')
else:
    print('✅ No conflicting columns (suffixes _x/_y) found.')
# --- Summary ---
print(f'Events:              {len(df_full)}')
print(f'Columns:             {len(df_full.columns)}')
print(f'Podcast cols dropped:{len(existing_to_drop)}')
print(f'played_frac min:     {df_full["played_frac"].min():.4f}')
print(f'played_frac max:     {df_full["played_frac"].max():.4f}')
print(f'Saved:               eda_ready.parquet')

print(df_full.dtypes)



✅ No conflicting columns (suffixes _x/_y) found.
Events:              79752
Columns:             49
Podcast cols dropped:0
played_frac min:     0.0000
played_frac max:     4.0331
Saved:               eda_ready.parquet
ts                    datetime64[ns]
platform                      object
ms_played                      int64
track_name                    object
artist_name                   object
album_name                    object
spotify_track_uri             object
reason_start                  object
reason_end                    object
shuffle                         bool
skipped                         bool
offline                         bool
account                       object
ts_utc                datetime64[ns]
year                           int32
month                          int32
day                            int32
hour                           int32
weekday                        int32
is_skip                         bool
release_date                  object
durat

## Содержание `00_1_build_full_history.ipynb`

**Цель ноутбука:**
Собрать и очистить историю прослушиваний, объединить с аудио-характеристиками из Exportify и сиротами, создать финальные `full_history.parquet` и `eda_ready.parquet`.

**Общие принципы:**
- Пути задаются в ячейке 1 через `src.paths` — никакого хардкода.
- Аккаунты определяются автоматически из структуры папок; в выводе имена аккаунтов не печатаются, только количество.
- Приватные колонки (`ip_addr`, `offline_timestamp`, `incognito_mode`, `conn_country`) удаляются в ячейке 2.
- Каждая ячейка читает/создаёт конкретный Parquet-файл.
- Промежуточные файлы сохраняются для воспроизводимости.
- Вывод ячеек — счётчики и `df.dtypes`, без таблиц с данными.

---

### Ячейка 1. Импорты, настройки, константы
**Что делает:**
- Импортирует библиотеки, добавляет корень репо в `sys.path` для импорта `src.*`.
- Загружает пути через `src.paths` (`get_project_root`, `get_data_path`, `get_base_path`, `get_genre_mapping_path`).
- Задаёт константы: `MIN_DURATION_MS`, `START_YEAR`, `END_YEAR`, пороги значимости сирот, `UTC_OFFSET_HOURS`.
- Собирает списки файлов: история (`Streaming_History_Audio_*.json`), Exportify (`spotify_playlists (*)/*.csv`), сироты (`exportify_orphan_export/*.csv`).
- Загружает `genre_mapping.json`.
- Определяет общие функции `has_cyrillic` и `normalize_genres` для всего ноутбука.

**Артефакты:** нет.

---

### Ячейка 2. Загрузка и очистка истории
**Вход:** `Streaming_History_Audio_*.json` по всем аккаунтам.
**Выход:** `cleaned_history.parquet`.

**Логика:**
- Читает все JSON, добавляет `account` (имя папки).
- Переименовывает поля в унифицированный формат (`endTime` → `ts` и т.д.).
- Сохраняет `ts_utc`, создаёт `ts = ts_utc + UTC_OFFSET_HOURS`.
- Производные: `year`, `month`, `day`, `hour`, `weekday`.
- Считает `is_skip = ms_played < 10_000 OR skipped == True`.
- Удаляет записи без названия/исполнителя и дубликаты по `(spotify_track_uri, ts, account)`.
- Фильтрует по годам `START_YEAR–END_YEAR`.
- Удаляет колонки подкастов/аудиокниг и **приватные** (`ip_addr`, `offline_timestamp`, `incognito_mode`, `conn_country`).
- Сохраняет файл.

---

### Ячейка 3. Загрузка и унификация Exportify
**Вход:** `spotify_playlists (*)/*.csv`.
**Выход:** `tracks_exportify.parquet`.

**Логика:**
- Читает все CSV, пропускает технический `.csv` (пустой stem) и пустые файлы.
- Добавляет `playlist_name`, `account`.
- Объединяет всё в `df_exportify_raw`.
- Удаляет дубликаты по `Track URI`.
- Переименовывает колонки в унифицированный формат.
- Удаляет строки с пустым `spotify_track_uri`.
- Заполняет пропуски в `genres` → `'unknown'`, нормализует через `genre_mapping`.
- Служебные поля: `has_audio_features=True`, `source='exportify'`.
- Приводит `Added By` к строке, пропуски → `'unknown'`.
- Сохраняет файл.

---

### Ячейка 3.1. Создание `playlist_links.parquet`
**Вход:** `df_exportify_raw` из ячейки 3.
**Выход:** `playlist_links.parquet`.

**Логика:**
- Берёт сырой датафрейм Exportify (все строки, до удаления дубликатов).
- Оставляет колонки: `Track URI`, `Added By`, `Added At`, `playlist_name`, `account`.
- Удаляет строки с пустым `Track URI`.
- Приводит `Added By` к строке, пропуски → `'unknown'`.
- Дубликаты не удаляет — поведенческие и технические повторы сохраняются для анализа.
- Сохраняет файл.

---

### Ячейка 4. Первое слияние: история + Exportify
**Вход:** `cleaned_history.parquet`, `tracks_exportify.parquet`.
**Выход:** `history_with_exportify.parquet`.

**Логика:**
- LEFT JOIN истории с треками по `spotify_track_uri`.
- Определяет `has_audio_features`: события с характеристиками и сироты.
- Заполняет пропуски в `track_name`, `artist_name`, `album_name` из данных трека.
- Удаляет колонки с суффиксом `_track`.
- Сохраняет файл.

---

### Ячейка 5. Загрузка эталонных сирот
**Вход:** CSV сирот из `exportify_orphan_export`.
**Выход:** `orphans_unified.parquet`.

**Логика:**
- Читает все части сирот, пропускает отсутствующие.
- Объединяет, удаляет дубликаты по `Track URI`.
- Переименовывает колонки в унифицированный формат.
- Оставляет только нужные колонки.
- Заполняет пропуски в жанрах `'unknown'`, нормализует.
- Служебные поля: `has_audio_features=True`, `source='exportify_orphans'`.
- Сохраняет файл.

---

### Ячейка 5.1. Проверка нормализации жанров сирот
**Вход:** `orphans_unified.parquet`.
**Выход:** нет (диагностика).

**Логика:**
- Считает долю `unknown` среди нормализованных жанров.
- Проверяет остаточную кириллицу — должна быть 0.
- Выводит топ-10 жанров среди сирот.

---

### Ячейка 6. Второе слияние: обогащение сирот
**Вход:** `history_with_exportify.parquet`, `orphans_unified.parquet`.
**Выход:** `history_fully_enriched.parquet`.

**Логика:**
- Определяет сирот как события с `has_audio_features != True`.
- Для сирот заполняет аудио-колонки из эталонной таблицы по URI.
- Обновляет `has_audio_features=True`, `source='exportify_orphans'`.
- Проверяет, что все события получили характеристики.
- Сохраняет файл.

---

### Ячейка 7. Категоризация сирот → `full_history.parquet`
**Вход:** `history_fully_enriched.parquet`.
**Выход:** `full_history.parquet`.

**Логика:**
- Удаляет `duration_ms == 0`, заменяет `tempo=0` и `popularity=0` на `NaN` — **до** категоризации.
- Считает метрики сирот по URI: `listen_count`, `max_played_ms`, `total_played_ms`, `first_listen`, `last_listen`.
- Помечает события без `valence`: `has_audio_features=False`, `orphan_category='unknown'`.
- Категории сирот с характеристиками:
  - `skipped` — если `is_skip`;
  - `significant` — `listen_count >= 3` или (`>= 2` и `played_frac >= 0.7`);
  - `insignificant` — остальные.
- Заполняет пропуски в жанрах.
- Удаляет лишние колонки (`playlist_name`, `Time Signature`, `account_track`).
- Сохраняет файл.
- Проверки: `first_listen` отсутствует у сирот — 0, `first_listen` у не-сирот — 0.

---

### Ячейка 8. Создание `eda_ready.parquet`
**Вход:** `full_history.parquet`.
**Выход:** `eda_ready.parquet`.

**Логика:**
- Загружает `full_history`.
- Добавляет `played_frac = ms_played / duration_ms`.
- Подстраховка: удаляет колонки подкастов/аудиокниг, если ещё остались.
- Сохраняет финальный `eda_ready.parquet`.

---

### Итог

После выполнения всех ячеек в `data/00_output/` появляются:

- `cleaned_history.parquet`
- `tracks_exportify.parquet`
- `playlist_links.parquet`
- `history_with_exportify.parquet`
- `orphans_unified.parquet`
- `history_fully_enriched.parquet`
- `full_history.parquet`
- `eda_ready.parquet`

Все файлы приватные, в git не попадают. Публичные агрегаты для дашборда собираются отдельно в `03_dashboard/dashboard/datasets/`.


## Содержание `00_1_build_full_history.ipynb`

**Цель ноутбука:**
Собрать историю прослушиваний, очистить, объединить с аудио-характеристиками из Exportify и с фичами сирот. На выходе — `eda_ready.parquet`, финальная таблица для EDA и дашборда.

**Общие принципы:**
- Пути через `src.paths`, без хардкода.
- Аккаунты определяются из структуры папок, в выводе — только счётчики.
- Приватные колонки (`ip_addr`, `offline_timestamp`, `incognito_mode`, `conn_country`) удаляются сразу.
- Каждая ячейка читает/создаёт конкретный parquet.
- Промежуточные файлы сохраняются.
- Вывод — счётчики и `df.dtypes`, без таблиц и путей.

---

### Ячейка 1. Импорты, настройки, константы

**Что делает:**
- Импортирует библиотеки, подключает `src/`.
- Загружает пути (`get_project_root`, `get_data_path`, `get_base_path`, `get_genre_mapping_path`).
- Задаёт константы: `MIN_DURATION_MS`, `START_YEAR`, `END_YEAR`, пороги значимости сирот, `UTC_OFFSET_HOURS`.
- Собирает списки файлов: история, Exportify, сироты.
- Загружает `genre_mapping.json`.
- Определяет функции `has_cyrillic` и `normalize_genres`.

**Выход:** переменные путей, списки файлов, функции.

---

### Ячейка 2. Загрузка и очистка истории

**Вход:** `Streaming_History_Audio_*.json` по всем аккаунтам.
**Выход:** `cleaned_history.parquet`.

**Логика:**
- Читает JSON, добавляет `account` (имя папки).
- Переименовывает поля Spotify → унифицированный формат (`endTime` → `ts`, `master_metadata_*` → `track_name`, `artist_name`, `album_name`).
- Сохраняет `ts_utc`, создаёт `ts = ts_utc + UTC_OFFSET_HOURS`.
- Производные: `year`, `month`, `day`, `hour`, `weekday`.
- Считает `is_skip = ms_played < 10_000 OR skipped == True`.
- Удаляет записи без названия/исполнителя и дубли по `(uri, ts, account)`.
- Фильтрует по годам `START_YEAR–END_YEAR`.
- Удаляет колонки подкастов и приватные.
- Сохраняет файл.

**Печатает:** сырых событий, удалённых, финальных строк, аккаунтов, диапазон лет.

---

### Ячейка 3. Загрузка и унификация Exportify

**Вход:** `spotify_playlists (*)/*.csv`.
**Выход:** `tracks_exportify.parquet`.

**Логика:**
- Читает все CSV, пропускает пустые и технические.
- Добавляет `playlist_name`, `account`.
- Дедупликация по `Track URI`.
- Переименование колонок (`Track URI` → `spotify_track_uri`, `Valence` → `valence`, и т.д.).
- Заполняет жанры, нормализует через `genre_mapping`.
- Служебные поля: `has_audio_features=True`, `source='exportify'`.
- `Added By` приводится к строке, пропуски → `'unknown'`.

**Печатает:** сколько CSV, удалено дублей, уникальных треков, % unknown жанров.

---

### Ячейка 3.1. Создание `playlist_links.parquet`

**Вход:** `df_exportify_raw` из ячейки 3.
**Выход:** `playlist_links.parquet`.

**Логика:**
- Берёт **сырой** Exportify (все строки, до удаления дублей).
- Оставляет только `Track URI`, `Added By`, `Added At`, `playlist_name`, `account`.
- Удаляет пустые URI.
- **Дубликаты не удаляет** — поведенческие повторы сохраняются для анализа.
- Приводит `Added By` к строке.

**Печатает:** строк, удалено, колонок.

---

### Ячейка 4. Первое слияние: история + Exportify

**Вход:** `cleaned_history.parquet`, `tracks_exportify.parquet`.
**Выход:** `history_with_exportify.parquet`.

**Логика:**
- LEFT JOIN истории с треками по `spotify_track_uri`.
- Определяет `has_audio_features`: события с фичами и сироты.
- Заполняет `track_name`, `artist_name`, `album_name` из треков, если пусто.
- Удаляет колонки с суффиксом `_track`.

**Печатает:** событий, треков, merged, с фичами, без фич.

---

### Ячейка 5. Загрузка эталонных сирот

**Вход:** CSV сирот из `exportify_orphan_export/`.
**Выход:** `orphans_unified.parquet`.

**Логика:**
- Читает все части сирот.
- Объединяет, дедупликация по `Track URI`.
- Переименование колонок в унифицированный формат.
- Оставляет только нужные колонки.
- Жанры заполняются `'unknown'`, нормализуются.
- Служебные поля: `has_audio_features=True`, `source='exportify_orphans'`.

**Печатает:** сколько частей, дублей, уникальных сирот, % unknown жанров.

---

### Ячейка 5.1. Проверка нормализации жанров сирот

**Вход:** `orphans_unified.parquet`.

**Логика:**
- Считает долю `unknown` среди нормализованных жанров.
- Проверяет остаточную кириллицу — должно быть 0.
- Топ-10 жанров среди сирот.

**Ничего не сохраняет — диагностика.**

---

### Ячейка 6. Второе слияние: обогащение сирот

**Вход:** `history_with_exportify.parquet`, `orphans_unified.parquet`.
**Выход:** `history_fully_enriched.parquet`.

**Логика:**
- Определяет сирот как события с `has_audio_features != True`.
- Для них заполняет аудио-колонки из эталонной таблицы по URI.
- Обновляет `has_audio_features=True`, `source='exportify_orphans'`.
- Проверяет, что все события получили характеристики.

**Печатает:** было сирот, осталось без фич.

---

### Ячейка 7. Категоризация сирот → `full_history.parquet`

**Вход:** `history_fully_enriched.parquet`.
**Выход:** `full_history.parquet`.

**Логика:**
- Удаляет `duration_ms == 0`, заменяет `tempo=0` и `popularity=0` на NaN.
- Считает метрики сирот по URI: `listen_count`, `max_played_ms`, `total_played_ms`, `first_listen`, `last_listen`.
- Помечает события без `valence`: `has_audio_features=False`, `orphan_category='unknown'`.
- Категории сирот: `skipped` / `significant` / `insignificant`.
- Заполняет пропуски в жанрах.
- Удаляет лишние колонки (`playlist_name`, `Time Signature`, `account_track`).
- Контроль: `first_listen` отсутствует у сирот — 0, у не-сирот — 0.

**Печатает:** всего событий, `orphan_category` в разрезе, NaN-чеки.

---

### Ячейка 8. Создание `eda_ready.parquet`

**Вход:** `full_history.parquet`.
**Выход:** `eda_ready.parquet`.

**Логика:**
- Загружает `full_history`.
- Добавляет `played_frac = ms_played / duration_ms`.
- Убирает колонки подкастов, если остались.
- Проверяет, что нет колонок с суффиксами `_x` / `_y` (артефакты merge).

**Печатает:** строк, колонок, диапазон `played_frac`, отсутствие конфликтов.

---

### Итог

После выполнения всех ячеек в `data/00_output/` появляются:

- `cleaned_history.parquet`
- `tracks_exportify.parquet`
- `playlist_links.parquet`
- `history_with_exportify.parquet`
- `orphans_unified.parquet`
- `history_fully_enriched.parquet`
- `full_history.parquet`
- `eda_ready.parquet`

Все паркеты приватные, в git не попадают.

---

## Что решает ноутбук

`00_1` решает задачу **сборки единой таблицы событий прослушивания с аудио-характеристиками**. Три источника данных лежат в разной форме: история знает что и когда слушалось, Exportify знает как звучат треки, сироты — треки вне плейлистов. Но ни один из них по отдельности не годится для анализа. Нужна таблица, где **на каждое событие** проставлены фичи его трека.

Главная сложность — масштаб. В истории 80 000 событий, но уникальных треков 25 000. Нельзя просто склеить таблицы: один трек может слушаться 50 раз, а метаданные нужны один раз на URI. Поэтому сначала все треки агрегируются в словарь «URI → фичи», потом через LEFT JOIN подставляются к каждому событию.

## Как решает

Последовательность из восьми шагов. **Очистка истории** — убираем подкасты, аудиокниги, приватные поля, дубли, короткие события. **Унификация Exportify** — собираем все CSV в одну таблицу уникальных треков с фичами. **Отдельно `playlist_links`** — там сохраняются все связи «плейлист ↔ трек», включая повторы (это поведенческий сигнал, его нельзя терять). **Первое слияние** — LEFT JOIN истории и треков. **Загрузка сирот** — отдельная таблица из CSV, которые подготовил `00_0`. **Второе слияние** — обогащение сирот фичами. **Категоризация** — определение, какие сироты значимые (`listen_count ≥ 3`), какие скипнутые, какие неизвестные. **Финальная сборка** — добавление `played_frac` и проверка целостности.

Ключевой момент — **JOIN по `spotify_track_uri`**. Это стабильный ID, одинаковый во всех источниках. Никаких fuzzy-матчей, никакой нормализации имён.

## Что даёт

Восемь паркетов в `data/00_output/`. Главный из них — **`eda_ready.parquet`** — таблица событий с фичами, готовая для EDA и дашборда. Плюс семь промежуточных файлов для отладки: если что-то сломалось, можно запустить с любого шага, не пересчитывая всё с нуля.

Ноутбук работает детерминированно — одинаковый вход даёт одинаковый выход. Сироты попадают в `full_history` со своими категориями (`significant` / `insignificant` / `skipped`), что позволяет дашборду отдельно анализировать значимые прослушивания и случайные касания. Проверки в ячейках 7 и 8 гарантируют, что ни один NaN не просочился в финальную таблицу.